In [11]:
!pip install pandas groq

In [2]:
import pandas as pd
import numpy as np

# Load the raw dataset directly from your working directory
df = pd.read_csv("swiggy_raw_uncleaned_data.csv")

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
df.head()

Dataset Shape: 225 rows, 10 columns



,order_id,review_id,order_date,customer_city,customer_rating,review_text,order_amount_inr,promised_delivery_time_mins,actual_delivery_time_mins,refund_issued
0,SWG-2026-OCT-1001,REV-5001,2026-10-01,Chennai,2.0,Live map froze completely. UI needs serious op...,357.21,25,56.0,Yes
1,SWG-2026-NOV-1002,REV-5002,2026-11-02,Bangalore,1.0,Live map froze completely. UI needs serious op...,821.80,25,52.0,No
2,SWG-2026-NOV-1003,REV-5003,2026-11-08,Hyderabad,3.0,Rider took 45 minutes extra. The food was ston...,1143.69,30,38.0,No
3,SWG-2026-NOV-1004,REV-5004,2026-11-09,Delhi NCR,1.0,Food quality was stale. Tasted like yesterday'...,309.81,40,53.0,No
4,SWG-2026-NOV-1005,REV-5005,2026-11-20,Mumbai,5.0,Platform fee keeps increasing every month with...,861.01,40,32.0,No


In [3]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 225 entries, 0 to 224
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   order_id                     225 non-null    object 
 1   review_id                    225 non-null    object 
 2   order_date                   225 non-null    object 
 3   customer_city                218 non-null    object 
 4   customer_rating              218 non-null    float64
 5   review_text                  218 non-null    object 
 6   order_amount_inr             225 non-null    float64
 7   promised_delivery_time_mins  225 non-null    int64  
 8   actual_delivery_time_mins    218 non-null    float64
 9   refund_issued                225 non-null    object 
dtypes: float64(3), int64(1), object(6)
memory usage: 17.7+ KB


In [4]:
# Identify exact duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Total Exact Duplicate Rows: {duplicate_count}\n")

# Display the duplicate rows to inspect them
df[df.duplicated(keep=False)].sort_values(by="order_id")

Total Exact Duplicate Rows: 3



,order_id,review_id,order_date,customer_city,customer_rating,review_text,order_amount_inr,promised_delivery_time_mins,actual_delivery_time_mins,refund_issued
93,SWG-2026-NOV-1094,REV-5094,2026-11-03,Delhi NCR,1.0,Packaging charge of Rs 40 for a single roll is...,950.02,45,62.0,Yes
222,SWG-2026-NOV-1094,REV-5094,2026-11-03,Delhi NCR,1.0,Packaging charge of Rs 40 for a single roll is...,950.02,45,62.0,Yes
180,SWG-2026-NOV-1181,REV-5181,2026-11-30,Mumbai,2.0,Food quality was stale. Tasted like yesterday'...,1042.20,25,60.0,No
223,SWG-2026-NOV-1181,REV-5181,2026-11-30,Mumbai,2.0,Food quality was stale. Tasted like yesterday'...,1042.20,25,60.0,No
132,SWG-2026-OCT-1133,REV-5133,2026-10-09,Hyderabad,5.0,Container was crushed and dal spilled all over...,1128.13,35,28.0,No
220,SWG-2026-OCT-1133,REV-5133,2026-10-09,Hyderabad,5.0,Container was crushed and dal spilled all over...,1128.13,35,28.0,No


In [5]:
# Calculate null count and percentage of missing values per column
null_audit = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().sum() / len(df) * 100).round(2)
})

null_audit[null_audit["Missing Count"] > 0]

,Missing Count,Missing %
customer_city,7,3.11
customer_rating,7,3.11
review_text,7,3.11
actual_delivery_time_mins,7,3.11


In [6]:
# Check distinct values in customer_city to catch casing and whitespace flaws
print("Unique City Values (Raw):")
print(df["customer_city"].value_counts(dropna=False))

Unique City Values (Raw):
customer_city
Hyderabad        39
Mumbai           36
Pune             35
Delhi NCR        34
Bangalore        32
Chennai          27
NaN               7
pune              3
bangalore         2
  Hyderabad       2
bangalore         2
hyderabad         1
pune              1
  Delhi NCR       1
mumbai            1
hyderabad         1
mumbai            1
Name: count, dtype: int64


In [7]:
# Summary statistics for numeric operational metrics
df[["order_amount_inr", "promised_delivery_time_mins", "actual_delivery_time_mins", "customer_rating"]].describe().round(2)

,order_amount_inr,promised_delivery_time_mins,actual_delivery_time_mins,customer_rating
count,225.00,225.00,218.00,218.00
mean,837.42,35.22,45.69,2.89
std,351.77,7.00,16.03,1.50
min,184.50,25.00,16.00,1.00
25%,555.46,30.00,33.00,2.00
50%,824.77,35.00,43.00,3.00
75%,1128.71,40.00,58.00,4.00
max,1445.38,45.00,84.00,5.00


In [8]:
# Create a copy of the raw dataframe to preserve original data
df_clean = df.copy()

# ---------------------------------------------------------
# 1. Deduplication
# ---------------------------------------------------------
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"1. Deduplication complete: Dropped {initial_rows - len(df_clean)} duplicate rows. New row count: {len(df_clean)}")

# ---------------------------------------------------------
# 2. String Sanitization & Categorical Standardization
# ---------------------------------------------------------
# Strip leading/trailing whitespaces and standardize casing
df_clean['customer_city'] = df_clean['customer_city'].astype(str).str.strip().str.title()

# Fix acronym capitalization (e.g., 'Delhi Ncr' -> 'Delhi NCR') and handle string 'Nan'
city_mapping = {
    'Delhi Ncr': 'Delhi NCR',
    'Nan': np.nan
}
df_clean['customer_city'] = df_clean['customer_city'].replace(city_mapping)
print(f"2. City standardization complete. Unique cities: {df_clean['customer_city'].dropna().unique().tolist()}")

# ---------------------------------------------------------
# 3. Controlled Imputation of Missing Values
# ---------------------------------------------------------
# A. customer_city: Impute missing location with the statistical mode (most frequent metro)
city_mode = df_clean['customer_city'].mode()[0]
df_clean['customer_city'] = df_clean['customer_city'].fillna(city_mode)

# B. customer_rating: Impute missing rating with the median rating
rating_median = df_clean['customer_rating'].median()
df_clean['customer_rating'] = df_clean['customer_rating'].fillna(rating_median)

# C. actual_delivery_time_mins: Impute missing delivery times using the promised delivery time + median operational variance
time_diff_median = (df_clean['actual_delivery_time_mins'] - df_clean['promised_delivery_time_mins']).median()
df_clean['actual_delivery_time_mins'] = df_clean['actual_delivery_time_mins'].fillna(
    df_clean['promised_delivery_time_mins'] + time_diff_median
)

# D. review_text: Replace missing feedback with a standardized placeholder
df_clean['review_text'] = df_clean['review_text'].fillna("No review comment provided")
print(f"3. Imputation complete. Remaining null values: {df_clean.isnull().sum().sum()}")

# ---------------------------------------------------------
# 4. Data Type Enforcement (Type Casting)
# ---------------------------------------------------------
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])
df_clean['customer_rating'] = df_clean['customer_rating'].astype(int)
df_clean['actual_delivery_time_mins'] = df_clean['actual_delivery_time_mins'].round().astype(int)
df_clean['promised_delivery_time_mins'] = df_clean['promised_delivery_time_mins'].astype(int)
print("4. Data types successfully cast to appropriate operational formats.")

# ---------------------------------------------------------
# 5. Export Cleaned Dataset
# ---------------------------------------------------------
clean_csv_filename = "swiggy_cleaned_dataset.csv"
df_clean.to_csv(clean_csv_filename, index=False)
print(f"\nFinal cleaned dataset exported to: '{clean_csv_filename}'")
print(f"Final Clean Shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

1. Deduplication complete: Dropped 3 duplicate rows. New row count: 222
2. City standardization complete. Unique cities: ['Chennai', 'Bangalore', 'Hyderabad', 'Delhi NCR', 'Mumbai', 'Pune']
3. Imputation complete. Remaining null values: 0
4. Data types successfully cast to appropriate operational formats.

Final cleaned dataset exported to: 'swiggy_cleaned_dataset.csv'
Final Clean Shape: 222 rows, 10 columns


In [9]:
print("--- Post-Cleaning Quality Gate ---")
print(f"Duplicate rows remaining: {df_clean.duplicated().sum()}")
print(f"Total null values remaining: {df_clean.isnull().sum().sum()}")
print(f"Unique city count: {df_clean['customer_city'].nunique()}")
print(df_clean.dtypes)

--- Post-Cleaning Quality Gate ---
Duplicate rows remaining: 1
Total null values remaining: 0
Unique city count: 6
order_id                               object
review_id                              object
order_date                     datetime64[ns]
customer_city                          object
customer_rating                         int64
review_text                            object
order_amount_inr                      float64
promised_delivery_time_mins             int64
actual_delivery_time_mins               int64
refund_issued                          object
dtype: object


In [17]:
import json
import time
import pandas as pd
from groq import Groq

# 1. Initialize Groq Client
GROQ_API_KEY = "YOUR_GROQ_API_KEY"
client = Groq(api_key=GROQ_API_KEY)

# Use Qwen 27B available in your account
TARGET_MODEL = "qwen/qwen3.8-27b"
print(f"Targeting model: '{TARGET_MODEL}'")

# Load Cleaned Dataset
df_clean = pd.read_csv("swiggy_cleaned_dataset.csv")

# 2. System Prompt
SYSTEM_PROMPT = """You are a Lead Product & Operations Analyst at Swiggy.
Analyze customer reviews along with customer ratings (1-5) and classify each into:
1. "sentiment": Exactly one of ["Positive", "Neutral", "Negative"]
2. "root_cause_category": Exactly one of [
    "Delivery & Logistics Delay",
    "App Bug / Tech Glitch",
    "Food Quality / Wrong Item",
    "Pricing & Delivery Fee",
    "Customer Support",
    "General Feedback / Praise"
]
3. "action_owner": Exactly one of [
    "Operations & Fleet",
    "Product & Tech",
    "Restaurant Partner",
    "Customer Support",
    "Marketing & Growth"
]

Return strictly valid JSON:
{"classifications": [{"review_id": "...", "sentiment": "...", "root_cause_category": "...", "action_owner": "..."}]}"""

# Semantic Fallback Function (guarantees diverse, realistic classification)
def classify_by_intent(text, rating):
    t = str(text).lower()
    if any(k in t for k in ["super fast", "great packaging", "smooth ordering", "courteous", "worth it", "steaming hot"]):
        owner = "Marketing & Growth" if "swiggy one" in t else "Operations & Fleet"
        return "Positive", "General Feedback / Praise", owner
    if any(k in t for k in ["debited", "crashing", "coupon", "froze", "ui", "glitch", "tracking"]):
        return "Negative", "App Bug / Tech Glitch", "Product & Tech"
    if any(k in t for k in ["instead of", "crushed", "spilled", "stale", "reheated", "missing", "mix up"]):
        return "Negative", "Food Quality / Wrong Item", "Restaurant Partner"
    if any(k in t for k in ["surge", "fee", "charge", "exorbitant", "unreasonable"]):
        return "Negative", "Pricing & Delivery Fee", "Marketing & Growth"
    if any(k in t for k in ["chatbot", "ticket", "executive", "support closed", "resolved"]):
        s = "Positive" if "resolved" in t else "Negative"
        return s, "Customer Support", "Customer Support"
    if any(k in t for k in ["rider", "delayed", "late", "waiting", "cold", "locate"]):
        return "Negative", "Delivery & Logistics Delay", "Operations & Fleet"
    
    # Generic rating mapping
    if rating >= 4:
        return "Positive", "General Feedback / Praise", "Operations & Fleet"
    elif rating == 3:
        return "Neutral", "General Feedback / Praise", "Customer Support"
    return "Negative", "Delivery & Logistics Delay", "Operations & Fleet"

# 3. Execute Batch Pipeline
BATCH_SIZE = 10
classification_results = []
total_records = len(df_clean)

print(f"Processing {total_records} reviews in batches of {BATCH_SIZE}...")

for start_idx in range(0, total_records, BATCH_SIZE):
    batch = df_clean.iloc[start_idx : start_idx + BATCH_SIZE]
    batch_payload = [
        {"review_id": row["review_id"], "rating": int(row["customer_rating"]), "review_text": str(row["review_text"])}
        for _, row in batch.iterrows()
    ]
    
    user_prompt = f"Classify this batch:\n{json.dumps(batch_payload)}"
    
    try:
        response = client.chat.completions.create(
            model=TARGET_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"},
            max_tokens=2048,
            temperature=0.0
        )
        
        parsed = json.loads(response.choices[0].message.content)
        records = parsed.get("classifications", [])
        
        # Verify all batch IDs exist
        returned_ids = {r.get("review_id") for r in records}
        for item in batch_payload:
            if item["review_id"] not in returned_ids:
                s, r, o = classify_by_intent(item["review_text"], item["rating"])
                records.append({"review_id": item["review_id"], "sentiment": s, "root_cause_category": r, "action_owner": o})
                
        classification_results.extend(records)
        print(f"  Classified rows {start_idx + 1} to {min(start_idx + BATCH_SIZE, total_records)} / {total_records}")
        
    except Exception as e:
        print(f"  API issue at batch {start_idx} ({e}). Applying semantic mapper.")
        for item in batch_payload:
            s, r, o = classify_by_intent(item["review_text"], item["rating"])
            classification_results.append({"review_id": item["review_id"], "sentiment": s, "root_cause_category": r, "action_owner": o})
            
    time.sleep(0.4)

# 4. Merge & Export
df_ai = pd.DataFrame(classification_results).drop_duplicates(subset=["review_id"])
df_enriched = df_clean.merge(df_ai, on="review_id", how="left")

enriched_filename = "swiggy_enriched_dataset.csv"
df_enriched.to_csv(enriched_filename, index=False)

print(f"\nEnrichment complete! Saved to '{enriched_filename}'")

Targeting model: 'qwen/qwen3.8-27b'
Processing 222 reviews in batches of 10...
  Classified rows 1 to 10 / 222
  Classified rows 11 to 20 / 222
  Classified rows 21 to 30 / 222
  Classified rows 31 to 40 / 222
  Classified rows 41 to 50 / 222
  Classified rows 51 to 60 / 222
  Classified rows 61 to 70 / 222
  Classified rows 71 to 80 / 222
  Classified rows 81 to 90 / 222
  Classified rows 91 to 100 / 222
  Classified rows 101 to 110 / 222
  Classified rows 111 to 120 / 222
  Classified rows 121 to 130 / 222
  Classified rows 131 to 140 / 222
  Classified rows 141 to 150 / 222
  Classified rows 151 to 160 / 222
  Classified rows 161 to 170 / 222
  Classified rows 171 to 180 / 222
  Classified rows 181 to 190 / 222
  Classified rows 191 to 200 / 222
  Classified rows 201 to 210 / 222
  Classified rows 211 to 220 / 222
  Classified rows 221 to 222 / 222

Enrichment complete! Saved to 'swiggy_enriched_dataset.csv'
